# Toronto Island Ferry Ticket Demand Forecasting & EDA
**Exploratory Data Analysis, Feature Engineering, Multi-Horizon Model Benchmarking, and Uncertainty Quantification**

This notebook analyzes the Toronto Island Ferry Ticket dataset (2015-2025) and builds short-term predictive forecasting models for Ticket Sales and Ticket Redemptions.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Setup paths
project_root = os.path.abspath(os.path.join('..', '.'))
if project_root not in sys.path:
    sys.path.append(project_root)

print('Environment initialized.')

## 1. Load Processed Clean Dataset
We load the 15-minute continuous series with 372,509 observations.

In [ ]:
clean_path = os.path.join(project_root, 'data', 'processed', 'clean_15min_timeseries.parquet')
df_clean = pd.read_parquet(clean_path)
print('Clean series shape:', df_clean.shape)
df_clean.head()

## 2. Diurnal Cycle & Day-of-Week EDA
Analyze 24-hour demand curves and weekend vs weekday surges.

In [ ]:
df_clean['hour'] = df_clean['Timestamp'].dt.hour
df_clean['dow'] = df_clean['Timestamp'].dt.day_name()
df_clean['is_weekend'] = df_clean['Timestamp'].dt.dayofweek >= 5

hourly_summary = df_clean.groupby(['hour', 'is_weekend'])[['Sales Count', 'Redemption Count']].mean().reset_index()
plt.figure(figsize=(10, 5))
sns.lineplot(data=hourly_summary, x='hour', y='Sales Count', hue='is_weekend', marker='o')
plt.title('Hourly Average Ticket Sales: Weekday vs Weekend')
plt.xlabel('Hour of Day')
plt.ylabel('Average Sales per 15-min')
plt.show()

## 3. Model Benchmark Results
Inspect the empirical evaluation metrics evaluated on the unseen 2025 holdout dataset.

In [ ]:
metrics_path = os.path.join(project_root, 'reports', 'results', 'model_evaluation_metrics.csv')
metrics_df = pd.read_csv(metrics_path)
display_cols = ['Target', 'Horizon', 'Model', 'MAE', 'RMSE', 'WAPE', 'Peak_Miss_Rate', 'Margin_90']
metrics_df[display_cols].sort_values(['Target', 'Horizon', 'MAE'])